# train_final_model.ipynb — Final YOLOv8 Training (Kaggle)

Great Barrier Reef COTS Detection Project

Wired to Person A's real data pipeline (`dataset.py`). Run cells top to
bottom, in order — each one prepares something the next one needs.

Running directly in the competition notebook — no Kaggle API auth or
download needed, data is already mounted at `/kaggle/input/`.

Turn on **Internet** in the Settings panel (right sidebar) so the
`git clone` and `pip install` cells below can reach GitHub/PyPI.
Also set **Accelerator → GPU T4 x2**.\n\n---\n**Updated:** now uses `yolo11s.pt` (was `yolov8n.pt`), trains for up to 100 epochs with early stopping (was a fixed 8), uses both GPUs, fixes an imgsz/optimizer mismatch from the previous run, and adds augmentation tuned for small underwater objects. See `model.py` for details.

## Step 1 — Get the repo code (`dataset.py`, `model.py`)

Clones fresh every run so you always get the latest pushed code.
Checks out your branch, then merges in `main` locally (inside this
session only — nothing gets pushed anywhere) so you have both your
files and anything Person A has pushed to `main`.

In [1]:
import sys

!rm -rf /kaggle/working/repo
!git clone https://github.com/RuzannaMkhitaryan/great-barrier-reef.git /kaggle/working/repo
!git -C /kaggle/working/repo checkout anahit
!git -C /kaggle/working/repo merge origin/main --no-edit

sys.path.append('/kaggle/working/repo/src')

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 78 (delta 25), reused 57 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 1.67 MiB | 22.80 MiB/s, done.
Resolving deltas: 100% (25/25), done.
Branch 'anahit' set up to track remote branch 'anahit' from 'origin'.
Switched to a new branch 'anahit'
Already up to date.


## Step 2 — Copy `splits.csv` into the working data folder

Adjust the source path below if it moves in the repo — check with
`!find /kaggle/working/repo -name splits.csv` if this cell errors.

In [2]:
!mkdir -p /kaggle/working/data
!cp /kaggle/working/repo/data/splits/splits.csv /kaggle/working/data/splits.csv

## Step 3 — Install dependencies

In [3]:
!pip install -q ultralytics

## Step 4 — Imports and config

In [4]:
!ls /kaggle/input

competitions


In [ ]:
import os
import glob
import shutil

import dataset
from model import load_model, get_train_config

# ---- CONFIG ----
# Competition data can be nested (e.g. /kaggle/input/competitions/<slug>/),
# so search recursively for train.csv instead of assuming a fixed depth.
_candidates = glob.glob("/kaggle/input/**/train.csv", recursive=True)
_reef_candidates = [c for c in _candidates if "barrier" in c.lower() or "reef" in c.lower()]
if _reef_candidates:
    _candidates = _reef_candidates
if not _candidates:
    raise FileNotFoundError("No train.csv under /kaggle/input/**. Run `!ls -R /kaggle/input` to check the Input panel, then set COMP_DIR manually.")
COMP_DIR = os.path.dirname(_candidates[0])
print(f"Using competition data at: {COMP_DIR}")

TRAIN_CSV = f"{COMP_DIR}/train.csv"
SPLITS_CSV = "/kaggle/working/data/splits.csv"
RAW_IMAGES_ROOT = f"{COMP_DIR}/train_images"

LABELS_TRAIN_DIR = "/kaggle/working/data/labels/train"
LABELS_VAL_DIR = "/kaggle/working/data/labels/val"
IMAGES_TRAIN_DIR = "/kaggle/working/data/images/train"
IMAGES_VAL_DIR = "/kaggle/working/data/images/val"
DATA_YAML_PATH = "/kaggle/working/data/data.yaml"

OUTPUT_DIR = "/kaggle/working/great-barrier-reef-checkpoints"
RUN_NAME = "final_model_v2"  # bumped from v1 so this run gets a clean folder instead of an auto -N suffix mixed in with old attempts

# Model checkpoint to start from. To try a different YOLO generation, just
# change this string - load_model() / Ultralytics handle the rest the same
# way for YOLOv8/v9/v10/YOLO11/YOLO26. Avoid yolo12*.pt (Ultralytics flags
# it as unstable to train). Bumped from yolov8n.pt (nano, 3M params) up to
# yolo11s.pt (small, ~9M params) for more capacity on this small-object task.
MODEL_WEIGHTS = "yolo11s.pt"

# Kaggle gave us GPU T4 x2 - actually use both instead of defaulting to one.
DEVICE = [0, 1]

# Person A's experiment found ratio=all (None) gave the best mAP50, but that
# was presumably measured with a long, converged training run. At only 8
# epochs we never converged, so ratio=None (67% background frames in train)
# mostly just slowed early learning. Start with ratio=2.0 for a faster real
# convergence check, then try None again once epochs/imgsz are fixed and you
# have time budget for a longer run.
NEGATIVE_RATIO = 2.0

## Step 5 — Dataset preparation functions

In [6]:
def clean_yolo_dataset():
    """
    Remove any previously generated YOLO dataset folders so every run
    starts clean - avoids stale labels/images from a prior ratio experiment
    leaking into the current run.
    """
    dirs_to_clean = [IMAGES_TRAIN_DIR, IMAGES_VAL_DIR, LABELS_TRAIN_DIR, LABELS_VAL_DIR]
    for directory in dirs_to_clean:
        if os.path.exists(directory):
            shutil.rmtree(directory)
        os.makedirs(directory, exist_ok=True)


def build_yolo_dataset(ratio=NEGATIVE_RATIO):
    """
    Runs Person A's full pipeline: load -> filter negatives (train only)
    -> write YOLO labels -> symlink images -> write data.yaml.
    Prepares files on disk for YOLO to read; returns nothing.
    """
    clean_yolo_dataset()

    full_df = dataset.load_data(TRAIN_CSV, SPLITS_CSV)
    train_df = full_df[full_df["split"] == "train"]
    val_df = full_df[full_df["split"] == "val"]  # never filtered - keep val untouched

    train_df_filtered = dataset.filter_negatives(train_df, ratio=ratio)

    print(f"Train frames after filtering: {len(train_df_filtered)} (ratio={ratio})")
    print(f"Val frames (untouched):       {len(val_df)}")

    dataset.write_yolo_labels(train_df_filtered, labels_dir=LABELS_TRAIN_DIR)
    dataset.write_yolo_labels(val_df, labels_dir=LABELS_VAL_DIR)

    dataset.link_images(train_df_filtered, RAW_IMAGES_ROOT, images_dir=IMAGES_TRAIN_DIR)
    dataset.link_images(val_df, RAW_IMAGES_ROOT, images_dir=IMAGES_VAL_DIR)

    dataset.write_data_yaml(DATA_YAML_PATH, IMAGES_TRAIN_DIR, IMAGES_VAL_DIR)

## Step 6 — Training function

In [ ]:
def train():
    build_yolo_dataset(ratio=NEGATIVE_RATIO)

    model = load_model(pretrained_weights=MODEL_WEIGHTS)

    # Previous run asked for imgsz=1280 and optimizer="SGD"/lr0=0.01, but the
    # Ultralytics log showed it actually trained at imgsz=960 with
    # optimizer='auto' -> AdamW(lr=0.002). Both settings are now threaded
    # through get_train_config(), and we print + verify the ACTUAL resolved
    # args after training starts so a mismatch like that is obvious
    # immediately instead of buried in the log.
    config = get_train_config(
        image_size=1280,
        epochs=100,
        batch_size=16,
        learning_rate=0.01,
        patience=20,
        optimizer="SGD",
        device=DEVICE,
    )
    print("Requested train config:", config)

    results = model.train(
        data=DATA_YAML_PATH,
        imgsz=config["imgsz"],
        epochs=config["epochs"],
        batch=config["batch"],
        optimizer=config["optimizer"],
        lr0=config["lr0"],
        patience=config["patience"],
        device=config["device"],
        mosaic=config["mosaic"],
        mixup=config["mixup"],
        copy_paste=config["copy_paste"],
        hsv_h=config["hsv_h"],
        hsv_s=config["hsv_s"],
        hsv_v=config["hsv_v"],
        project=OUTPUT_DIR,
        name=RUN_NAME,
        workers=4,
        cache="disk",
        plots=True,
    )

    # Sanity check: confirm what Ultralytics actually trained with matches
    # what we asked for. This would have caught the imgsz/optimizer mismatch
    # from the previous run immediately instead of requiring a log dig.
    actual = model.trainer.args
    print(f"Actual imgsz used:     {actual.imgsz} (requested {config['imgsz']})")
    print(f"Actual optimizer used: {actual.optimizer} (requested {config['optimizer']})")
    print(f"Actual lr0 used:       {actual.lr0} (requested {config['lr0']})")
    if actual.imgsz != config["imgsz"] or actual.optimizer != config["optimizer"]:
        print("\u26a0\ufe0f  Requested config does not match what Ultralytics actually used - investigate before trusting these results.")

    print("Training complete.")
    print(f"Results and checkpoints saved to: {OUTPUT_DIR}/{RUN_NAME}")
    return results

## Step 7 — Run training

Checkpoints save to `/kaggle/working/`, so they persist as notebook
output when you click **Save Version**.

In [ ]:
results = train()